# Experiments behind Submission 2 and Submission 3

FDS Kaggle challenge – Pokémon battle prediction (Gen 2 OU), Sapienza University of Rome.

This notebook documents **which experiments were run** to reach the two final models, and what they showed. It lives in the GitHub repo only: the Kaggle notebook just clones the repo and runs `make_submission_2.py` / `make_submission_3.py`, and does not use this file.

| | Submission 2 | Submission 3 |
| --- | --- | --- |
| Features | 73 features – `src/submission_2_features.py` | 79 features = the same 73 + 6 Snorlax features – `src/submission_3_features.py` |
| Model | Logistic Regression + CatBoost, soft voting, weights 1 : 1 (each model tuned separately) | Logistic Regression + CatBoost, soft voting, hyperparameters **and** weights tuned jointly |
| Script | `make_submission_2.py` | `make_submission_3.py` |
| 5-fold CV accuracy | 0.6837 | 0.6891 |


**Organisation**

1. Setup, data and the two feature sets
2. Logistic Regression: baseline and Optuna tuning
3. Gradient boosting baselines (XGBoost, LightGBM, CatBoost)
4. Tuning the gradient boosting models with Optuna
5. Ensembles of Logistic Regression + CatBoost (soft voting and stacking)
6. Joint Optuna tuning of the whole ensemble (Submission 3)
7. Summary

**Validation protocol.** Every score is a 5-fold stratified cross-validation accuracy (shuffled, seed 42) computed on `train.jsonl` only. The test set is never used to choose features or hyperparameters. All experiments use exactly the same folds, so the numbers are comparable.

**Recorded vs. re-run results.** The tables titled *Recorded results* are the outputs of the original exploratory runs. The code cells re-run the same experiments. The Optuna searches take hours, so they are switched off by default (`RUN_OPTUNA = False`); the notebook then evaluates the recorded best hyperparameters instead, which should reproduce the recorded scores (up to library versions). Except for the joint ensemble search (TPE sampler with seed 42), the searches were not seeded, so re-running them gives similar but not identical hyperparameters.

**Requirements:** `pip install numpy pandas scikit-learn optuna xgboost lightgbm catboost tqdm`. Run the notebook from inside the repo, with the competition files in `fds-pokemon-battles-prediction-2026/` (or on Kaggle).

## 0. Setup

In [ ]:
import os
import sys
import warnings

import numpy as np
import optuna
import pandas as pd
from catboost import CatBoostClassifier
from lightgbm import LGBMClassifier
from sklearn.ensemble import StackingClassifier, VotingClassifier
from sklearn.linear_model import LogisticRegression
from sklearn.model_selection import StratifiedKFold, cross_val_score
from sklearn.pipeline import Pipeline, make_pipeline
from sklearn.preprocessing import StandardScaler
from xgboost import XGBClassifier

# Make the repo's `src` package importable (the notebook may sit in a sub-folder of the repo)
REPO_ROOT = os.getcwd()
while not os.path.isdir(os.path.join(REPO_ROOT, "src")) and REPO_ROOT != os.path.dirname(REPO_ROOT):
    REPO_ROOT = os.path.dirname(REPO_ROOT)
sys.path.insert(0, REPO_ROOT)

from src.submission_2_features import build_feature_frames as build_features_sub2
from src.submission_2_features import load_jsonl, load_type_chart
from src.submission_3_features import build_feature_frames as build_features_sub3

# sklearn deprecates the `penalty=` argument; it is still used here exactly as in the original runs
warnings.filterwarnings("ignore", category=FutureWarning)
optuna.logging.set_verbosity(optuna.logging.WARNING)

In [ ]:
SEED = 42       # one seed for the CV splits, the joint Optuna sampler and every model
N_FOLDS = 5

# The Optuna searches below take hours. Leave this False to skip them and evaluate the recorded
# best hyperparameters instead; set it to True to re-run the searches.
RUN_OPTUNA = False

# Number of Optuna trials used in the original experiments
N_TRIALS = {
    "logreg_sub2": 100,     # Logistic Regression on the Submission 2 features: C and penalty (L1/L2)
    "logreg_sub3": 50,      # Logistic Regression on the Submission 3 features: C only (L2)
    "catboost": 50,
    "xgboost": 50,
    "lightgbm": 50,
    "joint_ensemble": 100,  # LR + CatBoost + voting weights, Submission 3 features
}

# Competition data: the Kaggle input folder if it exists, otherwise a local folder (edit if needed)
COMPETITION_DIR = "fds-pokemon-battles-prediction-2026"
KAGGLE_DATA_DIR = f"/kaggle/input/{COMPETITION_DIR}"
DATA_DIR = KAGGLE_DATA_DIR if os.path.exists(KAGGLE_DATA_DIR) else COMPETITION_DIR

# Hyperparameters found by the Optuna searches (recorded results, see sections 2, 4 and 6)
# Logistic Regression: best of the 100-trial search on the Submission 2 features (L2 penalty)
LR_TUNED_C = 0.019618501281982222

# CatBoost: best of the 50-trial search on the Submission 2 features
CATBOOST_TUNED = {
    "iterations": 1601,
    "learning_rate": 0.007622507163328224,
    "depth": 8,
    "l2_leaf_reg": 8.784785985319969,
    "random_strength": 0.7090587412583095,
    "border_count": 53,
}

# XGBoost and LightGBM: best of the 50-trial searches on the Submission 2 features
XGBOOST_TUNED = {
    "n_estimators": 1150,
    "learning_rate": 0.005364341235115622,
    "max_depth": 7,
    "min_child_weight": 8,
    "subsample": 0.581325989231801,
    "colsample_bytree": 0.8303164748455257,
    "gamma": 4.80414463593141,
    "reg_alpha": 0.0810963448591451,
    "reg_lambda": 2.7527113546995037,
}
LIGHTGBM_TUNED = {
    "n_estimators": 1041,
    "learning_rate": 0.023303753603654682,
    "num_leaves": 117,
    "max_depth": 3,
    "min_child_samples": 44,
    "subsample": 0.7094782100261967,
    "colsample_bytree": 0.57174905355602,
    "reg_alpha": 0.014218676188637213,
    "reg_lambda": 0.0047003983122060975,
}

# Joint search (Submission 3): best of 100 trials, identical to BEST_PARAMS in make_submission_3.py
JOINT_BEST_PARAMS = {
    "lr_c": 0.07830591333580003,
    "cb_iterations": 1970,
    "cb_lr": 0.01987056118032681,
    "cb_depth": 9,
    "cb_l2": 7.249997638789686,
    "cb_random_strength": 0.44843199034866,
    "cb_border_count": 240,
    "w_lr": 3.912985323019429,
    "w_cb": 2.726474219695041,
}

## 1. Data and features

Both submissions use the same 10,000 training battles. The **Submission 3 feature set is the Submission 2 feature set plus one extra group, the Snorlax features** (Snorlax HP, Attack boost and whether Curse was revealed, for both players): 6 extra features. The feature code is in `src/submission_2_features.py` (base features) and `src/submission_3_features.py` (adds the Snorlax group).

The only feature-set experiment recorded in the original notebooks is this comparison: **every experiment below was run on both feature sets** (except where noted), so each section also shows what the Snorlax features change.

The test battles are used only to build the pokédex (base stats and types of every Pokémon; no labels involved). They are never used to tune or select anything.

In [ ]:
train_data = load_jsonl(os.path.join(DATA_DIR, "train.jsonl"))
test_data = load_jsonl(os.path.join(DATA_DIR, "test.jsonl"))
type_chart = load_type_chart(os.path.join(DATA_DIR, "type_chart_gen2.csv"))
print(f"{len(train_data)} training battles, {len(test_data)} test battles")

train_df_2, _ = build_features_sub2(train_data, test_data, type_chart)   # Submission 2 features
train_df_3, _ = build_features_sub3(train_data, test_data, type_chart)   # Submission 3 features (+ Snorlax)

TARGET = "label_player_won"
FEATURES_2 = [c for c in train_df_2.columns if c not in ("battle_id", TARGET)]
FEATURES_3 = [c for c in train_df_3.columns if c not in ("battle_id", TARGET)]
X2, X3 = train_df_2[FEATURES_2], train_df_3[FEATURES_3]
y = train_df_2[TARGET]

assert y.equals(train_df_3[TARGET])            # same battles, same labels
assert set(FEATURES_2) < set(FEATURES_3)       # Submission 3 features = Submission 2 features + extras

SNORLAX_FEATURES = [c for c in FEATURES_3 if c not in FEATURES_2]
S2, S3 = "Sub2 feats (71)", "Sub3 feats (77)"
FEATURE_SETS = {S2: X2, S3: X3}

print(f"{S2}: {X2.shape}   {S3}: {X3.shape}")
print("Extra features of Submission 3:", SNORLAX_FEATURES)

### Helpers

All experiments share the same cross-validation splitter and the same evaluation function.

In [ ]:
CV = StratifiedKFold(n_splits=N_FOLDS, shuffle=True, random_state=SEED)
RESULTS = {}   # (experiment, feature set) -> mean CV accuracy of the runs in this notebook


def evaluate(model, X, label, experiment, feature_set):
    """5-fold CV accuracy of `model` on features `X` (same folds for every experiment)."""
    scores = cross_val_score(model, X, y, cv=CV, scoring="accuracy", n_jobs=-1)
    RESULTS[(experiment, feature_set)] = scores.mean()
    print(f"{label:<28} folds {np.round(scores, 4)}   mean {scores.mean():.4f} ± {scores.std():.4f}")
    return scores


def optimise(study_name, suggest_params, make_model, X, n_trials, sampler=None):
    """
    Generic Optuna search: maximise the mean 5-fold CV accuracy of `make_model(**suggest_params(trial))`.
    """
    def objective(trial):
        model = make_model(**suggest_params(trial))
        return cross_val_score(model, X, y, cv=CV, scoring="accuracy", n_jobs=-1).mean()

    study = optuna.create_study(direction="maximize", study_name=study_name, sampler=sampler)
    study.optimize(objective, n_trials=n_trials)
    return study


def report_study(name, study):
    print(f"{name}: best CV accuracy {study.best_value:.4f}")
    print(f"    {study.best_params}")

## 2. Logistic Regression

### 2.1 Baseline

`StandardScaler` + `LogisticRegression` (lbfgs solver, `C=0.114`, `max_iter=2500`).

**Recorded results**

| Features | 5-fold CV accuracy | Fold accuracies |
| --- | --- | --- |
| Submission 2 (73) | 0.6798 ± 0.0036 | 0.6795, 0.6810, 0.6730, 0.6825, 0.6830 |
| Submission 3 (79) | 0.6782 ± 0.0037 | 0.6780, 0.6825, 0.6715, 0.6800, 0.6790 |

In [ ]:
def logreg_baseline():
    return make_pipeline(
        StandardScaler(),
        LogisticRegression(C=0.114, solver="lbfgs", max_iter=2500, random_state=SEED),
    )


for fs_name, X in FEATURE_SETS.items():
    evaluate(logreg_baseline(), X, fs_name, "LR baseline", fs_name)

### 2.2 Tuning with Optuna

Search space: `C` in [1e-4, 1e2] (log scale), liblinear solver, `max_iter=2000`, objective = mean 5-fold CV accuracy.

- **Submission 2 features:** the penalty (L1 or L2) was searched as well, 100 trials.
- **Submission 3 features:** only `C` was searched (L2 penalty), 50 trials.

**Recorded results**

| Features | Trials | Best CV accuracy | Best parameters |
| --- | --- | --- | --- |
| Submission 2 (73) | 100 | 0.6812 (trial 29) | `C = 0.019618501281982222`, L2 penalty |
| Submission 3 (79) | 50 | 0.6789 | not saved |

Re-evaluating the best pipeline on the Submission 2 features gave 0.6812 ± 0.0030 (folds 0.6800, 0.6825, 0.6765, 0.6855, 0.6815). L2 won over L1, and this `C = 0.0196` is the Logistic Regression used in the ensembles of sections 5 (the best `C` of the Submission 3 search was not saved).

In [ ]:
def suggest_logreg(penalties):
    def suggest(trial):
        params = {"C": trial.suggest_float("C", 1e-4, 1e2, log=True)}
        if len(penalties) > 1:
            params["penalty"] = trial.suggest_categorical("penalty", penalties)
        return params
    return suggest


def make_logreg(**params):
    return Pipeline([
        ("scaler", StandardScaler()),
        ("classifier", LogisticRegression(**params, solver="liblinear", max_iter=2000, random_state=SEED)),
    ])


if RUN_OPTUNA:
    study = optimise("LogReg_Pokemon_Optimization", suggest_logreg(["l1", "l2"]), make_logreg, X2, N_TRIALS["logreg_sub2"])
    report_study(f"Logistic Regression, {S2}", study)
    study = optimise("LogReg_Pokemon_Optimization", suggest_logreg(["l2"]), make_logreg, X3, N_TRIALS["logreg_sub3"])
    report_study(f"Logistic Regression, {S3}", study)
else:
    print("RUN_OPTUNA = False: Logistic Regression search skipped (see the recorded results above).")

# Tuned Logistic Regression evaluated with the recorded C (used again in the ensembles below).
# Note: for the Submission 3 features this re-uses the C found on the Submission 2 features.
for fs_name, X in FEATURE_SETS.items():
    evaluate(make_logreg(C=LR_TUNED_C), X, fs_name, "LR tuned (C=0.0196)", fs_name)

## 3. Gradient boosting baselines

XGBoost, LightGBM and CatBoost with default hyperparameters (only the seed fixed), to see which library is the most promising before tuning.

**Recorded results** (5-fold CV accuracy)

| Model | Submission 2 features (73) | Submission 3 features (79) |
| --- | --- | --- |
| XGBoost | 0.6531 ± 0.0091 | 0.6567 ± 0.0046 |
| LightGBM | 0.6628 ± 0.0047 | 0.6733 ± 0.0074 |
| CatBoost | 0.6785 ± 0.0048 | 0.6754 ± 0.0021 |

CatBoost had the highest default accuracy on both feature sets.

In [ ]:
def baseline_gbms():
    return {
        "XGBoost": XGBClassifier(random_state=SEED, eval_metric="logloss"),
        "LightGBM": LGBMClassifier(random_state=SEED, verbose=-1),
        "CatBoost": CatBoostClassifier(random_state=SEED, verbose=0),
    }


for fs_name, X in FEATURE_SETS.items():
    print(fs_name)
    for model_name, model in baseline_gbms().items():
        evaluate(model, X, f"  {model_name}", f"{model_name} baseline", fs_name)

## 4. Tuning the gradient boosting models with Optuna

Each model was tuned separately with Optuna (50 trials, mean 5-fold CV accuracy as the objective). The search spaces are in the code below.

**Recorded results (Submission 2 features)**

| Model | Best CV accuracy (best trial) | Default accuracy (section 3) | Best hyperparameters |
| --- | --- | --- | --- |
| CatBoost | 0.6797 (trial 47) | 0.6785 | `iterations=1601, learning_rate=0.00762, depth=8, l2_leaf_reg=8.78, random_strength=0.709, border_count=53` |
| XGBoost | 0.6787 (trial 38) | 0.6531 | `n_estimators=1150, learning_rate=0.00536, max_depth=7, min_child_weight=8, subsample=0.581, colsample_bytree=0.830, gamma=4.80, reg_alpha=0.081, reg_lambda=2.75` |
| LightGBM | 0.6770 (trial 44) | 0.6628 | `n_estimators=1041, learning_rate=0.0233, num_leaves=117, max_depth=3, min_child_samples=44, subsample=0.709, colsample_bytree=0.572, reg_alpha=0.0142, reg_lambda=0.0047` |

(Full-precision values are in `CATBOOST_TUNED`, `XGBOOST_TUNED` and `LIGHTGBM_TUNED` in the setup cell.)

XGBoost and LightGBM were tuned on the Submission 2 features only. For the Submission 3 features the notebook contains the CatBoost search too, but its output was not saved; the Submission 3 ensemble in the original notebook uses exactly the same CatBoost and Logistic Regression hyperparameters as the Submission 2 ensemble.

Tuned scores are the best trial's CV mean (the same folds the search optimised), so they are slightly optimistic compared with the default scores of section 3.

In [ ]:
def suggest_catboost(trial):
    return {
        "iterations": trial.suggest_int("iterations", 500, 2000),
        "learning_rate": trial.suggest_float("learning_rate", 1e-3, 0.1, log=True),
        "depth": trial.suggest_int("depth", 4, 10),
        "l2_leaf_reg": trial.suggest_float("l2_leaf_reg", 1, 10),
        "random_strength": trial.suggest_float("random_strength", 0.1, 10.0, log=True),
        "border_count": trial.suggest_int("border_count", 32, 255),
    }


def suggest_xgboost(trial):
    return {
        "n_estimators": trial.suggest_int("n_estimators", 500, 2000),
        "learning_rate": trial.suggest_float("learning_rate", 1e-3, 0.1, log=True),
        "max_depth": trial.suggest_int("max_depth", 3, 9),
        "min_child_weight": trial.suggest_int("min_child_weight", 1, 10),
        "subsample": trial.suggest_float("subsample", 0.5, 1.0),
        "colsample_bytree": trial.suggest_float("colsample_bytree", 0.5, 1.0),
        "gamma": trial.suggest_float("gamma", 0, 5),
        "reg_alpha": trial.suggest_float("reg_alpha", 1e-5, 10, log=True),
        "reg_lambda": trial.suggest_float("reg_lambda", 1e-5, 10, log=True),
    }


def suggest_lightgbm(trial):
    return {
        "n_estimators": trial.suggest_int("n_estimators", 500, 2000),
        "learning_rate": trial.suggest_float("learning_rate", 1e-3, 0.1, log=True),
        "num_leaves": trial.suggest_int("num_leaves", 20, 150),
        "max_depth": trial.suggest_int("max_depth", 3, 12),
        "min_child_samples": trial.suggest_int("min_child_samples", 10, 100),
        "subsample": trial.suggest_float("subsample", 0.5, 1.0),
        "colsample_bytree": trial.suggest_float("colsample_bytree", 0.5, 1.0),
        "reg_alpha": trial.suggest_float("reg_alpha", 1e-5, 10, log=True),
        "reg_lambda": trial.suggest_float("reg_lambda", 1e-5, 10, log=True),
    }


def make_catboost(**params):
    return CatBoostClassifier(**params, eval_metric="Logloss", verbose=0, random_state=SEED)


def make_xgboost(**params):
    return XGBClassifier(**params, eval_metric="logloss", random_state=SEED, n_jobs=1)


def make_lightgbm(**params):
    return LGBMClassifier(**params, verbose=-1, random_state=SEED, n_jobs=1)


if RUN_OPTUNA:
    searches = [
        (f"CatBoost, {S2}", "CatBoost_Optimization", suggest_catboost, make_catboost, X2, N_TRIALS["catboost"]),
        (f"XGBoost, {S2}", "XGBoost_Optimization", suggest_xgboost, make_xgboost, X2, N_TRIALS["xgboost"]),
        (f"LightGBM, {S2}", "LightGBM_Optimization", suggest_lightgbm, make_lightgbm, X2, N_TRIALS["lightgbm"]),
        (f"CatBoost, {S3}", "CatBoost_Optimization", suggest_catboost, make_catboost, X3, N_TRIALS["catboost"]),
    ]
    for name, study_name, suggest, make, X, n_trials in searches:
        report_study(name, optimise(study_name, suggest, make, X, n_trials))
else:
    print("RUN_OPTUNA = False: gradient boosting searches skipped (see the recorded results above).")

# Re-evaluate the recorded best hyperparameters (found on the Submission 2 features)
tuned_gbms = {
    "CatBoost tuned": make_catboost(**CATBOOST_TUNED),
    "XGBoost tuned": make_xgboost(**XGBOOST_TUNED),
    "LightGBM tuned": make_lightgbm(**LIGHTGBM_TUNED),
}
for model_name, model in tuned_gbms.items():
    evaluate(model, X2, f"{model_name} ({S2})", model_name, S2)
evaluate(make_catboost(**CATBOOST_TUNED), X3, f"CatBoost tuned ({S3})", "CatBoost tuned", S3)

## 5. Ensembles of Logistic Regression + CatBoost

The tuned Logistic Regression (section 2.2) and the tuned CatBoost (section 4) were combined in two ways:

- **Soft voting with weights 1 : 1:** the two predicted probabilities are averaged. With the Submission 2 features this is **Submission 2**.
- **Stacking:** a Logistic Regression meta-model learns how to combine the two models' (5-fold out-of-fold) predictions.

**Recorded results** (5-fold CV accuracy)

| Ensemble | Submission 2 features (71) | Submission 3 features (77) |
| --- | --- | --- |
| Soft voting 1 : 1 | 0.6837 ± 0.0052 (folds 0.6875, 0.6860, 0.6735, 0.6865, 0.6850) | 0.6863 ± 0.0051 (folds 0.6905, 0.6900, 0.6765, 0.6865, 0.6880) |
| Stacking | 0.6838 ± 0.0052 (folds 0.6870, 0.6855, 0.6735, 0.6865, 0.6865) | output not saved |

For reference, the individual tuned models on the Submission 2 features scored 0.6812 (Logistic Regression) and 0.6797 (CatBoost), so blending the two adds about 0.003 – 0.004. Stacking scored the same as plain soft voting (0.6838 vs 0.6837).

In [ ]:
def lr_catboost_voting(weights=(1, 1)):
    return VotingClassifier(
        estimators=[("lr", make_logreg(C=LR_TUNED_C)), ("catboost", make_catboost(**CATBOOST_TUNED))],
        voting="soft",           # average the predicted probabilities
        weights=list(weights),
    )


def lr_catboost_stacking():
    return StackingClassifier(
        estimators=[("lr", make_logreg(C=LR_TUNED_C)), ("catboost", make_catboost(**CATBOOST_TUNED))],
        final_estimator=LogisticRegression(random_state=SEED),
        cv=5,
        n_jobs=-1,
    )


for fs_name, X in FEATURE_SETS.items():
    print(fs_name)
    evaluate(lr_catboost_voting(), X, "  soft voting 1:1", "LR+CatBoost soft voting 1:1", fs_name)
    evaluate(lr_catboost_stacking(), X, "  stacking", "LR+CatBoost stacking", fs_name)

## 6. Joint tuning of the whole ensemble (Submission 3)

Instead of tuning the two models separately and averaging them with equal weights, one Optuna search tuned **both models' hyperparameters and the two voting weights at the same time**, on the Submission 3 features:

- Logistic Regression: `C` in [1e-4, 10] (log)
- CatBoost: iterations 500 – 2000, learning rate 1e-3 – 0.1 (log), depth 4 – 10, `l2_leaf_reg` 0.1 – 15, `random_strength` 0.01 – 10, `border_count` 32 – 255
- Soft-voting weights `w_lr`, `w_cb` in [0.1, 5]
- Objective: mean 5-fold CV accuracy; TPE sampler with seed 42; **100 trials**

**Recorded results**

| | Value |
| --- | --- |
| Best trial | 61 of 100 |
| 5-fold CV accuracy of the best ensemble | **0.6891 ± 0.0036** (folds 0.6895, 0.6940, 0.6850, 0.6920, 0.6850) |
| Best hyperparameters | `lr_c=0.0783`, `cb_iterations=1970`, `cb_lr=0.0199`, `cb_depth=9`, `cb_l2=7.25`, `cb_random_strength=0.448`, `cb_border_count=240` |
| Best weights | `w_lr=3.91`, `w_cb=2.73` (about 59 % Logistic Regression, 41 % CatBoost) |

These values are the ones hard-coded in `make_submission_3.py`. A shorter exploratory re-run of the same search with only 10 trials reached 0.6832 ± 0.0054, so the number of trials mattered.

Compared with the separately tuned 1 : 1 ensemble on the same features (0.6863), joint tuning added about 0.003.

In [ ]:
def suggest_joint(trial):
    return {
        "lr_c": trial.suggest_float("lr_c", 1e-4, 10.0, log=True),
        "cb_iterations": trial.suggest_int("cb_iterations", 500, 2000),
        "cb_lr": trial.suggest_float("cb_lr", 0.001, 0.1, log=True),
        "cb_depth": trial.suggest_int("cb_depth", 4, 10),
        "cb_l2": trial.suggest_float("cb_l2", 0.1, 15.0),
        "cb_random_strength": trial.suggest_float("cb_random_strength", 0.01, 10.0),
        "cb_border_count": trial.suggest_int("cb_border_count", 32, 255),
        "w_lr": trial.suggest_float("w_lr", 0.1, 5.0),
        "w_cb": trial.suggest_float("w_cb", 0.1, 5.0),
    }


def build_joint_ensemble(params):
    """Soft-voting Logistic Regression + CatBoost ensemble from a dict of the 9 searched parameters."""
    lr = make_logreg(C=params["lr_c"])
    cb = make_catboost(
        iterations=params["cb_iterations"],
        learning_rate=params["cb_lr"],
        depth=params["cb_depth"],
        l2_leaf_reg=params["cb_l2"],
        random_strength=params["cb_random_strength"],
        border_count=params["cb_border_count"],
    )
    return VotingClassifier(
        estimators=[("lr", lr), ("catboost", cb)],
        voting="soft",
        weights=[params["w_lr"], params["w_cb"]],
        n_jobs=-1,
    )


if RUN_OPTUNA:
    joint_study = optimise(
        "Ensemble_Pokemon_Optimization",
        suggest_joint,
        build_joint_ensemble,
        X3,
        N_TRIALS["joint_ensemble"],
        sampler=optuna.samplers.TPESampler(seed=SEED),
    )
    report_study("Joint LR + CatBoost + weights", joint_study)
else:
    print("RUN_OPTUNA = False: joint search skipped (see the recorded results above).")

# Evaluate the recorded best parameters (= BEST_PARAMS in make_submission_3.py)
evaluate(build_joint_ensemble(JOINT_BEST_PARAMS), X3, S3, "Joint-tuned ensemble", S3)

## 7. Summary

**Recorded results of all experiments** (5-fold CV accuracy, mean ± std over the folds; "–" = not run or output not saved)

| Experiment | Submission 2 features (71) | Submission 3 features (77) |
| --- | --- | --- |
| Logistic Regression, baseline | 0.6798 ± 0.0036 | 0.6782 ± 0.0037 |
| Logistic Regression, tuned (Optuna) | 0.6812 ± 0.0030 | 0.6789 (best trial) |
| XGBoost, baseline | 0.6531 ± 0.0091 | 0.6567 ± 0.0046 |
| LightGBM, baseline | 0.6628 ± 0.0047 | 0.6733 ± 0.0074 |
| CatBoost, baseline | 0.6785 ± 0.0048 | 0.6754 ± 0.0021 |
| XGBoost, tuned (Optuna) | 0.6787 (best trial) | – |
| LightGBM, tuned (Optuna) | 0.6770 (best trial) | – |
| CatBoost, tuned (Optuna) | 0.6797 (best trial) | – |
| LR + CatBoost, soft voting 1 : 1 | **0.6837 ± 0.0052 → Submission 2** | 0.6863 ± 0.0051 |
| LR + CatBoost, stacking | 0.6838 ± 0.0052 | – |
| LR + CatBoost, joint Optuna search (weights tuned) | – | **0.6891 ± 0.0036 → Submission 3** |

**Final models.** The hyperparameters found here are hard-coded in `make_submission_2.py` and `make_submission_3.py`, which build the two submission files.

### Results of this run

The table below collects the scores computed by the cells above in the current session (the recorded results are in the sections above).

In [ ]:
experiment_order = list(dict.fromkeys(experiment for experiment, _ in RESULTS))
rerun_table = pd.Series(RESULTS).unstack().reindex(experiment_order).round(4)
print(rerun_table.to_string())